# Data Sharing: Atlas Charts

---

This notebook will prepare charts for vertex resolution spatial queries defined over the fs_LR surface template. (Showcasing that the pretrained SNM can be used for vertex-wise spatial queries)


### Package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
import colorcet as cc
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.image as mpimg
import joblib
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


In [4]:
from PIL import Image as PILImage
from IPython.display import display
import warnings

warnings.simplefilter('ignore', PILImage.DecompressionBombWarning)

def show_image(image_path, width=1000, resample=PILImage.LANCZOS):
    """
    Display an image in Jupyter at a fixed width while preserving aspect ratio.
    The image is resampled (not embedded full-size) to reduce notebook memory.
    """
    img = PILImage.open(image_path).copy()
    w, h = img.size
    new_h = int(h * width / w)
    img = img.resize((width, new_h), resample)
    display(img)


In [5]:
# use tex for plotting
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern']


### Brain visualization functions

---

Brain visualization scripts (utilizing [Cerebro Brain Viewer](https://cerebro-viewer.readthedocs.io/en/latest/))

In [6]:
# Cerebro brain viewer used for visualization
from cerebro import cerebro_brain_utils as cbu
from cerebro import cerebro_brain_viewer as cbv


In [7]:
# basic parameters
surface = 'pial'
expand = 0

# load an example dscalar
dscalar_file = cbu.cifti_template_file
dscalar = nib.load(dscalar_file)

brain_models = [x for x in dscalar.header.get_index_map(1).brain_models]

# load surfaces for visualization
left_surface_file, right_surface_file = cbu.get_left_and_right_GIFTI_template_surface(surface)
left_surface = nib.load(left_surface_file)
right_surface = nib.load(right_surface_file)

# extract surface information
lx, ly, lz = left_surface.darrays[0].data.T
lt = left_surface.darrays[1].data
rx, ry, rz = right_surface.darrays[0].data.T
rt = right_surface.darrays[1].data

# combine into a complete brain
lrx = np.concatenate([lx - expand, rx + expand])
lry = np.concatenate([ly, ry])
lrz = np.concatenate([lz, rz])
lrt = np.concatenate([lt, (rt + lx.shape[0])])

lxyz = left_surface.darrays[0].data
rxyz = right_surface.darrays[0].data
lrxyz = np.array([lrx, lry, lrz]).T

# create a mapping between surface and cifti vertices
left_cortical_surface_model, right_cortical_surface_model = brain_models[0], brain_models[1]
cifti_to_surface = {}
surface_to_cifti = {}
for (i, x) in enumerate(left_cortical_surface_model.vertex_indices):
    cifti_to_surface[i] = x
    surface_to_cifti[x] = i
for (i, x) in enumerate(right_cortical_surface_model.vertex_indices):
    cifti_to_surface[i + right_cortical_surface_model.index_offset] = x + rx.shape[0]
    surface_to_cifti[x + rx.shape[0]] = i + right_cortical_surface_model.index_offset

# construct data over surface
surface_mask = list(surface_to_cifti.keys())


In [8]:
# arbitrary colormap
mycm = LinearSegmentedColormap.from_list(
    'my_gradient',
    (
        (0.0, (0.1, 0.1, 1.,)),
        # (0.4999, (0.9, 0.9, 1.,)),
        (0.25, (0.1, 1., 1.,)),
        # (0.4999, (0.9, 1., 1.,)),
        (0.5, (1., 1., 1.,)),
        # (0.5001, (1., 1., 0.9,)),
        (0.75, (1., 1., 0.1,)),
        # (0.5001, (1., 0.9, 0.9,)),
        (1.0, (1., 0.1, 0.1,)),
    )
)

mycm_dark = LinearSegmentedColormap.from_list(
    'my_gradient',
    (
        (0.0, (0.05, 1., 1.,)),
        (0.35, (0.05, 0.05, 1.,)),
        (0.5, (0.2, 0.05, 0.05,)),
        (0.65, (1., 0.05, 0.05,)),
        (1.0, (1., 1., 0.05,)),
    )
)


In [9]:
# Brain visualizations with Cerebro

# ignore warning when loading cifti
nib.imageglobals.logger.setLevel(40)

# suppress Mesa/GL warnings
os.environ["LIBGL_DEBUG"] = "quiet"

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

def plot_single_view_with_cerebro(
        dscalar_data, ax, colormap=plt.cm.coolwarm, clims=None, vlims=None, exclusion_color=(1.,1.,1.,0),
        view="L", show_colorbar=False, colorbar_format=None, cifti_left_right_seperation=0,
        surface = 'midthickness', spheres=None):
    try:
        
        my_brain_viewer = cbv.Cerebro_brain_viewer(offscreen=True, background_color=(1.,1.,1.,0), null_color=(1., 1., 1., 0.0), no_color=(0.7, 0.7, 0.7, 1.))

        surface_model = my_brain_viewer.load_template_GIFTI_cortical_surface_models(surface)

        cifti_space = my_brain_viewer.visualize_cifti_space(
            volumetric_structures='none', cifti_left_right_seperation=cifti_left_right_seperation,
        )

        dscalar_layer = my_brain_viewer.add_cifti_dscalar_layer(
            dscalar_data=dscalar_data,
            colormap=colormap,
            clims=clims,
            vlims=vlims,
            exclusion_color=exclusion_color,
            opacity=0.95)
        
        if spheres is not None:
            my_brain_viewer.visualize_spheres(
                coordinates=spheres[0] + (spheres[2] @ np.array([[cifti_left_right_seperation/2, 0, 0]])),
                radii=1*np.ones(spheres[0].shape),
                color=spheres[1],
            )

        ax.axis('off')
        camconf = my_brain_viewer._view_to_camera_config(view)
        # camconf = my_brain_viewer.zoom_camera_to_content(camconf)
        my_brain_viewer.viewer.change_view(**camconf)
        my_brain_viewer.offscreen_draw_to_matplotlib_axes(ax)

    finally:
        my_brain_viewer.viewer.window.destroy()

def plot_left_right_surface_with_cerebro(dscalar_data, fig, ax, colormap=plt.cm.coolwarm, clims=None, show_colorbar=False, spheres=None, colorbar_format=None, **kwargs):
    # Hide the parent axis
    ax.set_visible(False)

    # Create a 2x2 GridSpec within the axis using inset_axes
    gs = ax.inset_axes([0, 0, 1, 1], transform=ax.transAxes)
    sub_gs = GridSpec(2, 2, gs, hspace=0., wspace=0.)

    # Create a 4x4 grid
    ax_tl = fig.add_subplot(sub_gs[0, 0])
    ax_tr = fig.add_subplot(sub_gs[0, 1])
    ax_bl = fig.add_subplot(sub_gs[1, 0])
    ax_br = fig.add_subplot(sub_gs[1, 1])
    
    # separate data to left and right
    dscalar_data_left = dscalar_data.copy()
    dscalar_data_left[left_cortical_surface_model.index_count:] = np.nan
    dscalar_data_right = dscalar_data.copy()
    dscalar_data_right[:left_cortical_surface_model.index_count] = np.nan
    
    # separate spheres to left and right
    if spheres is None:
        left_spheres = None
        right_spheres = None
    else:
        left_spheres_mask = (spheres[2][:,0] == -1)
        left_spheres = [spheres[0][left_spheres_mask], spheres[1], spheres[2][left_spheres_mask]]
        right_spheres_mask = (spheres[2][:,0] == 1)
        right_spheres = [spheres[0][right_spheres_mask], spheres[1], spheres[2][right_spheres_mask]]
    
    # Lateral left view
    plot_single_view_with_cerebro(dscalar_data_left, ax_tl, colormap=colormap, clims=clims,
                                  view=((-420, 0, 0), None, None, None), spheres=left_spheres, **kwargs)
    # Medial left view
    plot_single_view_with_cerebro(dscalar_data_left, ax_bl, colormap=colormap, clims=clims,
                                  view=((420, 0, 0), None, None, None), spheres=left_spheres, cifti_left_right_seperation=-80, **kwargs)
    
    # Lateral right view
    plot_single_view_with_cerebro(dscalar_data_right, ax_tr, colormap=colormap, clims=clims,
                                  view=((420, 0, 0), None, None, None), spheres=right_spheres, **kwargs)
    # Medial right view
    plot_single_view_with_cerebro(dscalar_data_right, ax_br, colormap=colormap, clims=clims,
                                  view=((-420, 0, 0), None, None, None), spheres=right_spheres, cifti_left_right_seperation=-80, **kwargs)

    if show_colorbar:
        cax = inset_axes(
            ax,
            width="30%",
            height="4%",
            loc="center",
            bbox_to_anchor=(-0., 0.0, 1., 1.),
            bbox_transform=ax.transAxes,
            borderpad=0,
        )
        cb = fig.colorbar(
            mpl.cm.ScalarMappable(
                norm=mpl.colors.Normalize(vmin=clims[0], vmax=clims[1]),
                cmap=colormap
            ),
            cax=cax,
            aspect=10,
            orientation='horizontal',
            format=colorbar_format,
        )
        cb.outline.set_visible(False)
        cb.ax.tick_params(labelsize=12)
        cb.ax.tick_params(length=0)


## Load Spectral Normative Model (SNM-1000)

---

Load the model from 09_01.


In [10]:
%%time
snm_1000 = snm.SpectralNormativeModel.load_model(
    "/mountpoint/code/projects/normative_brain_charts/data/pretrained_models/pretrained_SNM_1000_V1.0/"
)
eigenmode_basis = snm_1000.eigenmode_basis
snm_1000


CPU times: user 4.53 s, sys: 212 ms, total: 4.74 s
Wall time: 4.74 s


SpectralNormativeModel(eigenmode_basis=EigenmodeBasis(n_modes=1000, n_features=59412), base_model=DirectNormativeModel(spec=NormativeModelSpec(variable_of_interest='thickness', covariates=[CovariateSpec(name=age, cov_type=numerical, effect=spline), CovariateSpec(name=sex, cov_type=categorical, hierarchical=False, n_categories=2), CovariateSpec(name=site, cov_type=categorical, hierarchical=True, n_categories=189)], influencing_mean=['age', 'sex', 'site'], influencing_variance=['age', 'sex', 'site'])))

## Vertex-wise charting across lifespan

---



In [11]:
from Connectome_Spatial_Smoothing import CSS as css

smoothing_kernel_8mmFWHM_path = f'/mountpoint/code/projects/normative_brain_charts/data/smoothing_kernels/smoothing_kernel_8mmFWHM.npz'
if Path(smoothing_kernel_8mmFWHM_path).exists():
    smoothing_kernel_8mmFWHM = sparse.load_npz(smoothing_kernel_8mmFWHM_path)
else:
    # Create a smoothing kernel
    smoothing_kernel_8mmFWHM = css.compute_smoothing_kernel(left_surface_file, right_surface_file, fwhm=8, epsilon=0.01,)
    sparse.save_npz(ensure_dir(smoothing_kernel_8mmFWHM_path), smoothing_kernel_8mmFWHM)


In [12]:
%%time
high_res_norm_queries = eigenmode_basis.encode(smoothing_kernel_8mmFWHM.T, n_modes=1000)
high_res_norm_queries.shape


CPU times: user 3.46 s, sys: 719 ms, total: 4.18 s
Wall time: 4.18 s


(59412, 1000)

In [13]:
min_age, max_age = 5, 95
samples_per_year = 1
candidate_ages = np.linspace(min_age, max_age, samples_per_year * (max_age - min_age) + 1)


In [14]:
%%time

spectral_basis_predictions = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages,}),
    predict_without=['site', 'sex'],
)
spectral_basis_predictions_male = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages, "sex": ["M"]*len(candidate_ages),}),
    predict_without=['site'],
)
spectral_basis_predictions_female = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages, "sex": ["F"]*len(candidate_ages),}),
    predict_without=['site'],
)


Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

CPU times: user 7.91 s, sys: 789 ms, total: 8.7 s
Wall time: 16 s


In [15]:
%%time
# predict moments
predicted_moments = snm_1000.predict(
    encoded_query=high_res_norm_queries.T,
    spectral_predictions=spectral_basis_predictions,
    n_modes=1000,
).predictions

# save as joblib
# a python dictionary of means ('mu_estimate'), standard deviation estimates ('std_estimate'), with shapes of ages x vertices
# with an added key of ages ("Ages (years)")
predicted_moments["Ages (years)"] = candidate_ages
joblib.dump(predicted_moments, "/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.normative_trajectories.joblib")


CPU times: user 5min 52s, sys: 6min 38s, total: 12min 30s
Wall time: 5min 24s


['/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.normative_trajectories.joblib']

In [16]:
%%time
# predict moments
predicted_moments = snm_1000.predict(
    encoded_query=high_res_norm_queries.T,
    spectral_predictions=spectral_basis_predictions_male,
    n_modes=1000,
).predictions

# save as joblib
# a python dictionary of means ('mu_estimate'), standard deviation estimates ('std_estimate'), with shapes of ages x vertices
# with an added key of ages ("Ages (years)")
predicted_moments["Ages (years)"] = candidate_ages
joblib.dump(predicted_moments, "/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.male_normative_trajectories.joblib")


CPU times: user 6min 6s, sys: 6min 36s, total: 12min 43s
Wall time: 5min 37s


['/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.male_normative_trajectories.joblib']

In [17]:
%%time
# predict moments
predicted_moments = snm_1000.predict(
    encoded_query=high_res_norm_queries.T,
    spectral_predictions=spectral_basis_predictions_female,
    n_modes=1000,
).predictions

# save as joblib
# a python dictionary of means ('mu_estimate'), standard deviation estimates ('std_estimate'), with shapes of ages x vertices
# with an added key of ages ("Ages (years)")
predicted_moments["Ages (years)"] = candidate_ages
joblib.dump(predicted_moments, "/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.female_normative_trajectories.joblib")


CPU times: user 5min 48s, sys: 6min 36s, total: 12min 24s
Wall time: 5min 17s


['/mountpoint/code/projects/normative_brain_charts/data/charts/vertex-wise.fs_LR_32k.female_normative_trajectories.joblib']